# Linear Algebra 2 — Matrices and Transformations

This notebook builds directly on Linear Algebra 1. Where that week was about vectors, this week is about **matrices** — rectangular grids of numbers that represent linear transformations.

By the end of this notebook you will understand:
- How to read and work with a matrix
- How to multiply a matrix by a vector
- The three fundamental 2D transformations: translation, rotation, and scaling
- How to combine transformations by multiplying their matrices together

## Setup
Run this cell first — it loads all the drawing and maths helpers used throughout the notebook.

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Polygon

# ── Drawing helpers ───────────────────────────────────────────────────────────

def draw_vector(ax, vec, color='steelblue', label='', start=(0, 0), lw=2):
    ax.annotate('', xy=(start[0]+vec[0], start[1]+vec[1]), xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        ax.text(start[0]+vec[0]*0.55, start[1]+vec[1]*0.55,
                label, color=color, fontsize=11, fontweight='bold')

def setup_axes(ax, xlim=(-1, 5), ylim=(-1, 5)):
    ax.set_xlim(*xlim);  ax.set_ylim(*ylim)
    ax.axhline(0, color='#ccc', lw=0.8)
    ax.axvline(0, color='#ccc', lw=0.8)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

# ── Maths helpers ─────────────────────────────────────────────────────────────

def mat_vec(M, v):
    """Multiply matrix M by column vector v."""
    return [sum(M[i][j] * v[j] for j in range(len(v))) for i in range(len(M))]

def mat_mul(A, B):
    """Multiply matrix A by matrix B."""
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]

def rotation_matrix(theta):
    c, s = math.cos(theta), math.sin(theta)
    return [[c, -s], [s, c]]

def scale_matrix(sx, sy):
    return [[sx, 0], [0, sy]]

def translation_matrix(tx, ty):
    """3x3 homogeneous translation matrix."""
    return [[1, 0, tx], [0, 1, ty], [0, 0, 1]]

def apply_transform(M, points):
    """Apply 2x2 matrix M to a list of 2D points."""
    return [mat_vec(M, p) for p in points]

def draw_shape(ax, points, color, label='', alpha=1.0, lw=2, linestyle='-'):
    closed = points + [points[0]]
    xs = [p[0] for p in closed]
    ys = [p[1] for p in closed]
    ax.plot(xs, ys, color=color, lw=lw, alpha=alpha, linestyle=linestyle, label=label)

print('Ready.')

---
## 1. Matrices

A **matrix** is a rectangular grid of numbers arranged in rows and columns. A matrix with *m* rows and *n* columns is called an *m × n matrix*.

In Python we store a matrix as a **list of rows**, where each row is itself a list:

```
M = [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]
```

Access element at row `r`, column `c` with `M[r][c]` (both **0-indexed**).

The most important matrix in linear algebra is the **identity matrix** — it leaves every vector unchanged, much like multiplying a number by 1:

$$I = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$

**Why matrices matter:**

- **Computer graphics:** every rotation, scale, or camera move in a 3D game is encoded as a matrix. Applying the matrix to a vertex's position vector gives the transformed position.
- **Machine learning:** the weights of a neural network layer are stored as a matrix. A forward pass is a series of matrix-vector multiplications.
- **Image processing:** convolution filters (blur, sharpen, edge-detect) are small matrices slid over an image.

In [ ]:
M = [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]]

print('The matrix M:')
for row in M:
    print(' ', row)

print(f'\nM[0][0] = {M[0][0]}  ← top-left')
print(f'M[1][2] = {M[1][2]}  ← row 1, col 2')
print(f'M[2][1] = {M[2][1]}  ← row 2, col 1')

# Build and display the 2x2 identity matrix
I = [[1, 0], [0, 1]]
print('\nIdentity matrix:', I)

### Viewing a matrix as column vectors

One of the most useful ways to think about a 2×2 matrix is as **two column vectors** that tell you where the standard basis vectors `e₁ = [1, 0]` and `e₂ = [0, 1]` end up after the transformation.

$$M = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$$

- First column `[a, c]` — the image of **e₁**
- Second column `[b, d]` — the image of **e₂**

Once you know where those two basis vectors go, you know where **everything** goes.

In [ ]:
M = [[2, 1],
     [0, 1]]

# Where do the basis vectors land?
e1 = [M[0][0], M[1][0]]   # first column
e2 = [M[0][1], M[1][1]]   # second column

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before
ax = axes[0]
draw_vector(ax, [1, 0], color='steelblue',  label='e₁ = [1,0]')
draw_vector(ax, [0, 1], color='salmon',      label='e₂ = [0,1]')

# Lightly draw the unit square
sq = [[0,0],[1,0],[1,1],[0,1]]
draw_shape(ax, sq, '#aaa', alpha=0.5, linestyle='--')
setup_axes(ax, xlim=(-1, 4), ylim=(-1, 3))
ax.set_title('Before: standard basis vectors', fontsize=11)

# After
ax = axes[1]
draw_vector(ax, e1, color='steelblue', label=f'M·e₁ = {e1}')
draw_vector(ax, e2, color='salmon',    label=f'M·e₂ = {e2}')

# Transformed unit square
tsq = apply_transform(M, sq)
draw_shape(ax, tsq, '#aaa', alpha=0.5, linestyle='--')
setup_axes(ax, xlim=(-1, 4), ylim=(-1, 3))
ax.set_title(f'After M = {M}: where basis vectors land', fontsize=11)

plt.suptitle('A matrix as a pair of column vectors', fontsize=13)
plt.tight_layout();  plt.show()

print(f'M = {M}')
print(f'Column 1 (image of e₁): {e1}')
print(f'Column 2 (image of e₂): {e2}')

---
## 2. Matrix-Vector Multiplication

Multiplying a matrix **M** by a vector **v** produces a new vector. You compute each component of the output by taking the **dot product of that row of M with v**:

$$(Mv)_i = \sum_j M_{ij} \cdot v_j$$

For a 2×2 matrix:

$$\begin{bmatrix} a & b \\ c & d \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} ax + by \\ cx + dy \end{bmatrix}$$

**Geometric meaning:** this is how the matrix *acts* on a vector — it transforms the vector to a new position.

**Real-world examples:**

- **3D rendering:** every vertex of a 3D model is a vector. Multiplying by the camera matrix converts world coordinates to screen coordinates. This happens millions of times per second in any 3D application.
- **Neural networks:** a fully-connected layer computes `output = M · input + bias`. With a batch of inputs you do one big matrix multiplication — the GPU's whole job is doing this extremely fast.
- **Recommender systems:** Netflix/Spotify represent each user and each item as a vector of hidden features. A matrix multiplication across all users at once produces every recommendation score.

In [ ]:
M = [[2, 0],
     [1, 3]]
v = [4, 2]
Mv = mat_vec(M, v)

print(f'M = {M}')
print(f'v = {v}')
print(f'Mv = [{M[0][0]}×{v[0]} + {M[0][1]}×{v[1]},  {M[1][0]}×{v[0]} + {M[1][1]}×{v[1]}]')
print(f'Mv = {Mv}')

fig, ax = plt.subplots(figsize=(6, 6))
draw_vector(ax, v,  color='steelblue', label=f'v = {v}')
draw_vector(ax, Mv, color='seagreen',  label=f'Mv = {Mv}')

# Show the row-dot-product construction lines
ax.annotate('', xy=(Mv[0], 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='salmon', lw=1.5, linestyle='dashed'))
ax.annotate('', xy=(Mv[0], Mv[1]), xytext=(Mv[0], 0),
            arrowprops=dict(arrowstyle='->', color='mediumpurple', lw=1.5, linestyle='dashed'))
ax.text(Mv[0]/2, -0.4, f'row 0·v = {Mv[0]}', color='salmon',       fontsize=9)
ax.text(Mv[0]+0.1, Mv[1]/2, f'row 1·v = {Mv[1]}', color='mediumpurple', fontsize=9)

setup_axes(ax, xlim=(-1, 10), ylim=(-1, 12))
ax.set_title('Matrix-vector multiplication: each row dotted with v', fontsize=11)
plt.tight_layout();  plt.show()

### Visualising the full transformation

Apply **M** to every point in a grid to see what the matrix *does* to space.

In [ ]:
M = [[2, 0.5],
     [0, 1  ]]

# Build a grid of points
grid = [[x, y] for x in range(-3, 4) for y in range(-3, 4)]
transformed = apply_transform(M, grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, points, title in [(axes[0], grid, 'Before transform'),
                           (axes[1], transformed, f'After M = {M}')]:
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    ax.scatter(xs, ys, s=18, color='steelblue', alpha=0.7)
    setup_axes(ax, xlim=(-7, 7), ylim=(-4, 4))
    ax.set_title(title, fontsize=11)

plt.suptitle('Grid of points before and after M (shear + scale)', fontsize=13)
plt.tight_layout();  plt.show()

---
## 3. Linear Transformations

A **linear transformation** maps every point in space to a new point while preserving:
1. **Lines stay lines** — no curves, no bends.
2. **The origin stays at the origin.**

Every linear transformation can be represented by a matrix, and applying the matrix is the same as applying the transformation.

The three fundamental 2D transformations are **translation**, **rotation**, and **scaling**. We will look at each in turn.

### 3a. Translation

Translation moves every point by a fixed offset `(tx, ty)`. Alone, this is **not** a linear transformation (it moves the origin) — but we can make it one by using **homogeneous coordinates**.

Encode a 2D point `[x, y]` as a 3D vector `[x, y, 1]`, and use the 3×3 matrix:

$$T = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}$$

Then $T \cdot [x, y, 1]^\top = [x + t_x,\ y + t_y,\ 1]^\top$.

Drop the trailing 1 to read off the translated 2D point.

**Real-world use:** homogeneous coordinates are used universally in computer graphics (OpenGL, DirectX, WebGL) so that translation, rotation, and scale can all be combined into one matrix multiplication.

In [ ]:
# A simple square as a set of 2D points
square = [[0, 0], [2, 0], [2, 2], [0, 2]]
tx, ty = 3, 1

def translate(points, tx, ty):
    return [[p[0] + tx, p[1] + ty] for p in points]

moved = translate(square, tx, ty)

fig, ax = plt.subplots(figsize=(7, 5))
draw_shape(ax, square, 'steelblue', label='original', lw=2)
draw_shape(ax, moved,  'seagreen',  label='translated', lw=2)

# Show the translation arrow from centre to centre
cx1 = sum(p[0] for p in square) / len(square)
cy1 = sum(p[1] for p in square) / len(square)
ax.annotate('', xy=(cx1 + tx, cy1 + ty), xytext=(cx1, cy1),
            arrowprops=dict(arrowstyle='->', color='crimson', lw=2, linestyle='dashed'))
ax.text(cx1 + tx/2, cy1 + ty/2 + 0.3, f'(+{tx}, +{ty})', color='crimson', fontsize=11)

# Show the 3x3 homogeneous matrix
T = translation_matrix(tx, ty)
p_h = [1, 1, 1]   # point [1,1] in homogeneous coords
result_h = mat_vec(T, p_h)

setup_axes(ax, xlim=(-1, 8), ylim=(-1, 5))
ax.legend(fontsize=10)
ax.set_title(f'Translation by (tx={tx}, ty={ty})', fontsize=12)
plt.tight_layout();  plt.show()

print(f'T = {T}')
print(f'Translating point [1, 1] via homogeneous coords:')
print(f'  T · [1, 1, 1] = {result_h}  ← drop the 1: ({result_h[0]}, {result_h[1]})')

### 3b. Rotation

A **rotation matrix** rotates every point in the plane counter-clockwise by angle θ around the origin:

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Key angles to remember:

| θ | cos θ | sin θ | Effect |
|---|---|---|---|
| 0° | 1 | 0 | no change (identity) |
| 90° | 0 | 1 | 90° counter-clockwise |
| 180° | −1 | 0 | flip (point reflection) |
| 270° | 0 | −1 | 90° clockwise |

**Real-world use:** rotating 3D models and cameras in games; animating characters; rotating images; defining coordinate frames in robotics.

In [ ]:
angles_deg = [0, 45, 90, 135]
colors      = ['steelblue', 'salmon', 'seagreen', 'mediumpurple']
arrow       = [[0, 0], [2, 0], [2, 0.5], [2.8, 0], [2, -0.5], [2, 0]]  # simple arrow shape

fig, ax = plt.subplots(figsize=(6, 6))

for deg, col in zip(angles_deg, colors):
    theta = math.radians(deg)
    R = rotation_matrix(theta)
    rotated_shape = apply_transform(R, arrow)
    draw_shape(ax, rotated_shape, col, label=f'{deg}°', lw=2)

# Unit circle reference
thetas = [i * 2 * math.pi / 360 for i in range(361)]
ax.plot([math.cos(t)*2.8 for t in thetas], [math.sin(t)*2.8 for t in thetas],
        'k--', lw=0.6, alpha=0.2)

setup_axes(ax, xlim=(-4, 4), ylim=(-4, 4))
ax.legend(title='Rotation angle', fontsize=10)
ax.set_title('Rotation: the same arrow at 0°, 45°, 90°, 135°', fontsize=11)
plt.tight_layout();  plt.show()

# Check: rotating e1 = [1,0] by 90° should give [0,1]
R90 = rotation_matrix(math.pi / 2)
e1  = [1, 0]
result = mat_vec(R90, e1)
print(f'R(90°) · [1, 0] = [{result[0]:.4f}, {result[1]:.4f}]  ← should be [0, 1]')

### 3c. Scaling

A **scale matrix** stretches or compresses space along the x and y axes:

$$S = \begin{bmatrix} s_x & 0 \\ 0 & s_y \end{bmatrix}$$

- `sx > 1`: stretch horizontally — e.g. `sx = 2` doubles all x-coordinates.
- `sx < 1`: compress horizontally — e.g. `sx = 0.5` halves all x-coordinates.
- `sx = sy`: **uniform** scale — preserves shape (aspect ratio).
- `sx ≠ sy`: **non-uniform** scale — stretches the shape.
- Negative values flip the shape about an axis.

**Real-world use:** resizing images or textures; zoom effects in games; normalising input features in machine learning (scaling every feature to the range [0, 1]).

In [ ]:
# A unit square
unit_sq = [[0, 0], [1, 0], [1, 1], [0, 1]]

configs = [
    (2, 2,   'salmon',       'uniform scale (×2)'),
    (3, 1,   'seagreen',     'horizontal stretch (sx=3, sy=1)'),
    (1, 0.5, 'mediumpurple', 'vertical squash (sx=1, sy=0.5)'),
]

fig, ax = plt.subplots(figsize=(8, 6))
draw_shape(ax, unit_sq, 'steelblue', label='original (1×1)', lw=2, linestyle='--')

for sx, sy, col, lbl in configs:
    S = scale_matrix(sx, sy)
    scaled = apply_transform(S, unit_sq)
    draw_shape(ax, scaled, col, label=lbl, lw=2)

setup_axes(ax, xlim=(-0.5, 4), ylim=(-0.5, 3))
ax.legend(fontsize=9, loc='upper right')
ax.set_title('Scaling: stretching and squashing the unit square', fontsize=11)
plt.tight_layout();  plt.show()

v = [3, 2]
sx, sy = 2, 3
S = scale_matrix(sx, sy)
print(f'scale({sx}, {sy}) · {v} = {mat_vec(S, v)}')

---
## 4. Combining Transformations: Matrix Multiplication

To apply transformation A *then* transformation B, you could do:
1. `v2 = A · v`, then `v3 = B · v2`

But you can do the same thing in one step by first **multiplying the matrices**:

$$C = B \cdot A \quad \Rightarrow \quad v_3 = C \cdot v$$

> **Order matters.** Matrix multiplication is not commutative: `B · A ≠ A · B` in general.  
> By convention, transforms are applied **right-to-left**: in `B · A`, A happens first.

The formula for element `C[i][j]` is the dot product of **row i of B** with **column j of A**:

$$C_{ij} = \sum_k B_{ik} \cdot A_{kj}$$

**Real-world use:** graphics pipelines pre-multiply the model, view, and projection matrices into a single *MVP matrix*, then apply it to every vertex once per frame — saving enormous computation.

In [ ]:
# Compose: scale by 2 in x, then rotate 45°
S   = scale_matrix(2, 1)
R45 = rotation_matrix(math.radians(45))

# Combined matrix: R45 · S  (scale first, then rotate)
C = mat_mul(R45, S)

print('Scale matrix S:')
for row in S:
    print(' ', [round(x, 4) for x in row])

print('Rotation matrix R45:')
for row in R45:
    print(' ', [round(x, 4) for x in row])

print('Combined C = R45 · S:')
for row in C:
    print(' ', [round(x, 4) for x in row])

# Verify: apply via C vs applying S then R45
v = [1, 0]
via_combined = mat_vec(C, v)
via_steps    = mat_vec(R45, mat_vec(S, v))

print(f'\nv = {v}')
print(f'C · v               = {[round(x, 4) for x in via_combined]}')
print(f'R45 · (S · v)       = {[round(x, 4) for x in via_steps]}')
print(f'Results match: {all(abs(a-b)<1e-9 for a,b in zip(via_combined, via_steps))}')

In [ ]:
# Visualise: scale then rotate vs rotate then scale
shape = [[0, 0], [2, 0], [2, 1], [0, 1]]

S   = scale_matrix(2, 0.5)
R90 = rotation_matrix(math.pi / 2)

# S first, then R90
scale_then_rotate = apply_transform(mat_mul(R90, S), shape)
# R90 first, then S
rotate_then_scale = apply_transform(mat_mul(S, R90), shape)

fig, ax = plt.subplots(figsize=(8, 6))
draw_shape(ax, shape,             'steelblue',     label='original', lw=2, linestyle='--')
draw_shape(ax, scale_then_rotate, 'seagreen',      label='scale → rotate', lw=2)
draw_shape(ax, rotate_then_scale, 'mediumpurple',  label='rotate → scale', lw=2)

setup_axes(ax, xlim=(-3, 5), ylim=(-5, 4))
ax.legend(fontsize=10)
ax.set_title('Order matters: scale→rotate ≠ rotate→scale', fontsize=11)
plt.tight_layout();  plt.show()

### Worked example: verify A @ B

Compute $C = A \cdot B$ step by step:

$$A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}, \quad B = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}$$

$$C_{00} = 1 \times 5 + 2 \times 7 = 5 + 14 = 19 \qquad C_{01} = 1 \times 6 + 2 \times 8 = 6 + 16 = 22$$

$$C_{10} = 3 \times 5 + 4 \times 7 = 15 + 28 = 43 \qquad C_{11} = 3 \times 6 + 4 \times 8 = 18 + 32 = 50$$

In [ ]:
A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
C = mat_mul(A, B)

print(f'A = {A}')
print(f'B = {B}')
print(f'A @ B = {C}')

# Show the element-by-element calculation
print('\nBreakdown:')
for i in range(2):
    for j in range(2):
        terms = ' + '.join(f'{A[i][k]}×{B[k][j]}' for k in range(2))
        print(f'  C[{i}][{j}] = {terms} = {C[i][j]}')

---
## Summary

| Concept | Key idea | Matrix |
|---|---|---|
| **Matrix** | Grid of numbers; columns show where basis vectors land | `M[r][c]` |
| **Mat-vec multiply** | Each output row = dot of that row with v | `(Mv)[i] = Σ M[i][j]·v[j]` |
| **Translation** | Shift by (tx, ty) — needs homogeneous coords | 3×3: `[[1,0,tx],[0,1,ty],[0,0,1]]` |
| **Rotation** | Counter-clockwise by θ | `[[cosθ, −sinθ],[sinθ, cosθ]]` |
| **Scaling** | Stretch/compress per axis | `[[sx, 0],[0, sy]]` |
| **Matrix multiplication** | Compose transforms; order matters | `C = A @ B` (B applied first) |

### What's next?

Now open **exercises.py** and fill in the six TODOs. Run `python check.py` from the repository root to see how you did. The A-Frame 3D visualisations in `la2-aframe/` let you explore each concept interactively in your browser.